# Imports

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint

In [2]:
def compute_metrics(df):
    """
    For high-diff pairs: agreement with gold label (accuracy + Cohen's Kappa).
    For low-diff pairs: test for absence of systematic preference (binomial test + Wilson CI).
    Returns two DataFrames: high_df and low_df.
    """
    high_metrics = []
    low_metrics = []

    for kpi in sorted(df["kpi_id"].unique()):

        # --- HIGH DIFF: agreement between human choice and gold label ---
        subset = df[(df["kpi_id"] == kpi) & (df["diff_level"] == "high")]
        accuracy = accuracy_score(subset["left_better"], subset["left_chosen"])
        kappa = cohen_kappa_score(subset["left_better"], subset["left_chosen"])
        high_metrics.append({
            "kpi_id": kpi,
            "n_pairs": len(subset),
            "accuracy": round(accuracy, 4),
            "cohen_kappa": round(kappa, 4),
        })

        # --- LOW DIFF: test whether human choice deviates from chance (50/50) ---
        subset = df[(df["kpi_id"] == kpi) & (df["diff_level"] == "low")]
        n = len(subset)
        n_left = int(subset["left_chosen"].sum())
        prop = n_left / n
        binom_p = binomtest(n_left, n, p=0.5).pvalue
        ci_low, ci_high = proportion_confint(n_left, n, method="wilson")
        low_metrics.append({
            "kpi_id": kpi,
            "n_pairs": n,
            "prop_left_chosen": round(prop, 4),
            "wilson_ci_low": round(ci_low, 4),
            "wilson_ci_high": round(ci_high, 4),
            "binomial_p": round(binom_p, 4),
            "random_choice": "yes (p>0.05)" if binom_p > 0.05 else "no (p≤0.05)",
        })

    high_df = pd.DataFrame(high_metrics)
    low_df = pd.DataFrame(low_metrics)
    return high_df, low_df

# Evaluator 1

In [ ]:
# Load data
df_1 = pd.read_excel(r"mini_exp_revised_evaluator_1.xlsx")
df_1

In [4]:
high_df_1, low_df_1 = compute_metrics(df_1)

print("=== Evaluator 1 | High-diff: Agreement with gold (refined vs. initial) ===")
display(high_df_1)

print("\n=== Evaluator 1 | Low-diff: Test for no systematic preference ===")
display(low_df_1)

=== Evaluator 1 | High-diff: Agreement with gold (refined vs. initial) ===


,kpi_id,n_pairs,accuracy,cohen_kappa
0,1,80,0.8875,0.7750
1,2,80,0.8375,0.6738
2,3,80,0.9125,0.8250



=== Evaluator 1 | Low-diff: Test for no systematic preference ===


,kpi_id,n_pairs,prop_left_chosen,wilson_ci_low,wilson_ci_high,binomial_p,random_choice
0,1,80,0.4250,0.3226,0.5343,0.2185,yes (p>0.05)
1,2,80,0.5000,0.3930,0.6070,1.0000,yes (p>0.05)
2,3,80,0.6375,0.5281,0.7343,0.0183,no (p≤0.05)


# Evaluator 2

In [5]:
df_2 = pd.read_excel(r"mini_exp_revised_evaluator_2.xlsx")

high_df_2, low_df_2 = compute_metrics(df_2)

print("=== Evaluator 2 | High-diff: Agreement with gold (refined vs. initial) ===")
display(high_df_2)

print("\n=== Evaluator 2 | Low-diff: Test for no systematic preference ===")
display(low_df_2)

=== Evaluator 2 | High-diff: Agreement with gold (refined vs. initial) ===


,kpi_id,n_pairs,accuracy,cohen_kappa
0,1,80,0.7500,0.5000
1,2,80,0.7750,0.5361
2,3,80,0.7625,0.5250



=== Evaluator 2 | Low-diff: Test for no systematic preference ===


,kpi_id,n_pairs,prop_left_chosen,wilson_ci_low,wilson_ci_high,binomial_p,random_choice
0,1,80,0.4250,0.3226,0.5343,0.2185,yes (p>0.05)
1,2,80,0.5500,0.4412,0.6542,0.4340,yes (p>0.05)
2,3,80,0.4625,0.3575,0.5710,0.5764,yes (p>0.05)


# Together

In [ ]:
df_all = pd.concat([df_1, df_2], ignore_index=True)
df_all.head()

In [7]:
high_df_all, low_df_all = compute_metrics(df_all)

print("=== Combined | High-diff: Agreement with gold (refined vs. initial) ===")
display(high_df_all)

print("\n=== Combined | Low-diff: Test for no systematic preference ===")
display(low_df_all)

=== Combined | High-diff: Agreement with gold (refined vs. initial) ===


,kpi_id,n_pairs,accuracy,cohen_kappa
0,1,160,0.8187,0.6375
1,2,160,0.8063,0.6058
2,3,160,0.8375,0.6750



=== Combined | Low-diff: Test for no systematic preference ===


,kpi_id,n_pairs,prop_left_chosen,wilson_ci_low,wilson_ci_high,binomial_p,random_choice
0,1,160,0.425,0.3510,0.5025,0.0687,yes (p>0.05)
1,2,160,0.525,0.4479,0.6009,0.5801,yes (p>0.05)
2,3,160,0.550,0.4726,0.6250,0.2356,yes (p>0.05)
